# Ames House Price Prediction — Preprocessing

This notebook compares preprocessing strategies using the same 80/20 train/holdout
split (`random_state=42`) as the Data Understanding and EDA notebooks.

- Remove the identifier `Id` from predictors.
- Treat `MSSubClass` as a categorical code rather than a continuous number.
- Fit imputation, encoding, scaling, and regression **inside each cross-validation
  training fold** using scikit-learn pipelines.
- Use only training-set cross-validation for experimentation; the holdout set is
  reserved for final evaluation.

The target is modeled on the `log1p(SalePrice)` scale; reported RMSE scores are
in **log-price units**, not dollars or an external competition score.


In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
housing = fetch_openml(data_id=42165, as_frame=True)

X_all = housing.data.drop(columns=["Id"]).copy()
y_all = pd.to_numeric(housing.target).copy()


X_all["MSSubClass"] = X_all["MSSubClass"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42
)

y_train_log = np.log1p(y_train)

assert "Id" not in X_train.columns
assert X_train.index.intersection(X_test.index).empty
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

print("Training features:", X_train.shape)
print("Holdout features:", X_test.shape)
print("Training target:", y_train.shape)


Training features: (1168, 79)
Holdout features: (292, 79)
Training target: (1168,)


In [3]:
numeric_features = X_train.select_dtypes(
    include="number"
).columns

categorical_features = X_train.select_dtypes(
    exclude="number"
).columns

In [4]:
assert "Id" not in numeric_features
assert "MSSubClass" in categorical_features
assert len(numeric_features) + len(categorical_features) == X_train.shape[1]

print(f"Numerical features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features: {X_train.shape[1]}")


Numerical features: 35
Categorical features: 44
Total features: 79


## Baseline Missing-Value Handling

We first establish a simple preprocessing baseline before comparing more
informed strategies:

- Fill missing numerical values with the **training-fold median**.
- Encode missing categorical values as a dedicated `"Missing"` category.
- One-hot encode categorical predictors, ignoring unseen categories at inference.

The baseline deliberately does not yet distinguish *structural absence* (such as
`NoGarage`) from unknown data. That distinction may be introduced in later,
controlled experiments. All imputation parameters are fitted within CV folds.


In [5]:
numeric_imputer = SimpleImputer(
    strategy="median"
)

In [6]:
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Missing"
)


In [7]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore"
)

In [8]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", numeric_imputer)
    ]
)

In [9]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", categorical_imputer),
        ("encoder", categorical_encoder)
    ]
)

### Preprocessing Pipelines

The numerical pipeline imputes missing values using medians, while the
categorical pipeline adds a `"Missing"` category and one-hot encodes the result.
When paired with an estimator in the outer `Pipeline`, each transformation is
fitted only on the corresponding training fold during cross-validation.


In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

### Combining Numerical and Categorical Preprocessing

A `ColumnTransformer` applies different preprocessing pipelines to different
groups of columns.

- Numerical features are processed using the numerical pipeline.
- Categorical features are processed using the categorical pipeline.
- The transformed outputs are then combined into a single feature matrix.

This allows all preprocessing rules to be managed in one object while preserving
the appropriate treatment for each feature type.

In [11]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression(tol=1e-10))
    ]
)

## Baseline Model

A linear regression model is used as the first baseline.

The preprocessing pipeline and model are combined into a single pipeline so that
all preprocessing steps are fitted independently within each cross-validation
training fold.

Model performance is evaluated using RMSE on the log-transformed target.

In [12]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
cv_rmse = -cross_val_score(
    baseline_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

cv_rmse

array([0.14561579, 0.15738696, 0.2098075 , 0.14486321, 0.16925191])

In [14]:
cv_rmse.mean(), cv_rmse.std()

(np.float64(0.1653850755964647), np.float64(0.023934264785417856))

### Baseline Result

The baseline's five cross-validation RMSE values, mean, and standard deviation
are shown above. These are **log1p-price RMSE** scores; lower is better.

This baseline provides the reference for the experiments below. Numerical
results must be read from the current execution rather than old saved scores.


## Experiment 1 — Numerical Missing Indicators

The baseline median imputation removes information about whether a numerical
value was originally missing.

As a controlled preprocessing experiment, missing-value indicators are added
to numerical features while keeping all other preprocessing and model settings
unchanged.

This allows the model to use missingness itself as a potential predictive signal.

In [15]:
numeric_imputer_with_indicator = SimpleImputer(
    strategy="median",
    add_indicator=True
)

In [16]:
numeric_pipeline_with_indicator = Pipeline(
    steps=[
        ("imputer", numeric_imputer_with_indicator)
    ]
)

In [17]:
preprocessor_with_indicator = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_with_indicator,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [18]:
model_with_indicator = Pipeline(
    steps=[
        ("preprocessor", preprocessor_with_indicator),
        ("model", LinearRegression(tol=1e-10))
    ]
)

In [19]:
cv_rmse_with_indicator = -cross_val_score(
    model_with_indicator,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

cv_rmse_with_indicator

array([0.14516083, 0.15751975, 0.21054229, 0.14498628, 0.16932374])

### Numerical Missing Indicator Result

Numerical missing-value indicators let the model learn whether an observation
was originally missing, rather than only using its median-imputed value.

Compare the mean and fold-to-fold variation shown in the next cell with the
baseline's values. Retain the additional indicators only if their improvement
is meaningful relative to CV variability and model complexity.


In [20]:
cv_rmse_with_indicator.mean(), cv_rmse_with_indicator.std()

(np.float64(0.1655065786213284), np.float64(0.024255338132593646))

## Experiment 2 — Neighborhood-Based LotFrontage Imputation

Exploratory analysis suggested that `LotFrontage` varies between neighborhoods.
This experiment replaces the global frontage median with a **neighborhood
median learned within each CV training fold**.

If a neighborhood has no available learned median, the transform falls back to
the global median from the **same training fold**. The custom transformer returns
just the imputed `LotFrontage` column; `Neighborhood` is also processed separately
by the categorical pipeline.


In [21]:
class NeighborhoodLotFrontageImputer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        required = {"Neighborhood", "LotFrontage"}
        if not required.issubset(X.columns):
            raise ValueError("Neighborhood and LotFrontage columns are required.")

        self.neighborhood_medians_ = (
            X.groupby("Neighborhood")["LotFrontage"].median()
        )
        self.global_median_ = X["LotFrontage"].median()
        if pd.isna(self.global_median_):
            raise ValueError("Cannot impute LotFrontage: no observed training values.")
        return self

    def transform(self, X):
        if not {"Neighborhood", "LotFrontage"}.issubset(X.columns):
            raise ValueError("Neighborhood and LotFrontage columns are required.")
        X = X.copy()

        neighborhood_fill_values = (
            X["Neighborhood"]
            .map(self.neighborhood_medians_)
            .fillna(self.global_median_)
        )
        X["LotFrontage"] = X["LotFrontage"].fillna(
            neighborhood_fill_values
        )
        return X[["LotFrontage"]]


In [22]:
numeric_features_without_lot_frontage = (
    numeric_features.drop("LotFrontage")
)

In [23]:
lot_frontage_imputer = NeighborhoodLotFrontageImputer()

In [24]:
preprocessor_with_neighborhood_frontage = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features_without_lot_frontage
        ),
        (
            "lot_frontage",
            lot_frontage_imputer,
            ["LotFrontage", "Neighborhood"]
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [25]:
model_with_neighborhood_frontage = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_with_neighborhood_frontage
        ),
        (
            "model",
            LinearRegression(tol=1e-10)
        )
    ]
)

In [26]:
cv_rmse_neighborhood_frontage = -cross_val_score(
    model_with_neighborhood_frontage,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

cv_rmse_neighborhood_frontage

array([0.14564181, 0.15736547, 0.21014516, 0.14425252, 0.16952456])

In [27]:
cv_rmse_neighborhood_frontage.mean(), \
cv_rmse_neighborhood_frontage.std()

(np.float64(0.1653859028481018), np.float64(0.024171456896231968))

In [28]:
experiment_results = pd.DataFrame({
    "experiment": [
        "Baseline",
        "Numerical Missing Indicators",
        "Neighborhood LotFrontage"
    ],
    "mean_cv_rmse": [
        cv_rmse.mean(),
        cv_rmse_with_indicator.mean(),
        cv_rmse_neighborhood_frontage.mean()
    ],
    "std_cv_rmse": [
        cv_rmse.std(),
        cv_rmse_with_indicator.std(),
        cv_rmse_neighborhood_frontage.std()
    ]
})

experiment_results.sort_values("mean_cv_rmse")

,experiment,mean_cv_rmse,std_cv_rmse
0,Baseline,0.165385,0.023934
2,Neighborhood LotFrontage,0.165386,0.024171
1,Numerical Missing Indicators,0.165507,0.024255


### Neighborhood-Based LotFrontage Result

The table above compares the baseline, numerical missing indicators, and
neighborhood-based `LotFrontage` imputation on identical CV folds.

A lower mean RMSE favors a method, but small differences should be interpreted
with the standard deviation and individual folds in mind. Neighborhood-based
imputation is a candidate, not an established improvement until the revised
results are evaluated.


## Experiment 3 — Scaled Linear Regression and Ridge

Some numerical predictors are strongly correlated. Ridge regression is tested
as a regularized alternative to ordinary linear regression.

Numerical variables are standardized **within CV folds** before penalized
regression so coefficients on differently scaled measurements receive a more
comparable penalty. `MSSubClass` is instead one-hot encoded as categorical.
The experiment also checks how the unregularized linear model behaves after
scaling.


In [29]:
numeric_pipeline_scaled = Pipeline(
    steps=[
        ("imputer", numeric_imputer),
        ("scaler", StandardScaler())
    ]
)

In [30]:
lot_frontage_pipeline_scaled = Pipeline(
    steps=[
        (
            "imputer",
            NeighborhoodLotFrontageImputer()
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [31]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_scaled,
            numeric_features_without_lot_frontage
        ),
        (
            "lot_frontage",
            lot_frontage_pipeline_scaled,
            ["LotFrontage", "Neighborhood"]
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [32]:
linear_scaled_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", LinearRegression(tol=1e-10))
    ]
)

In [33]:
cv_rmse_linear_scaled = -cross_val_score(
    linear_scaled_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

cv_rmse_linear_scaled.mean(), cv_rmse_linear_scaled.std()

(np.float64(0.16852412525454413), np.float64(0.026301155341015316))

In [34]:
ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", Ridge(alpha=1.0))
    ]
)

In [35]:
cv_rmse_ridge = -cross_val_score(
    ridge_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

cv_rmse_ridge

array([0.14116488, 0.14070444, 0.20121528, 0.12289997, 0.13363513])

In [36]:
cv_rmse_ridge.mean(), cv_rmse_ridge.std()

(np.float64(0.14792393991625316), np.float64(0.02745229403318652))

### Ridge Regression Result

Compare the scaled ordinary linear regression and `Ridge(alpha=1.0)` CV scores
with the earlier pipelines above. If scaling changes ordinary least squares
performance substantially, numerical conditioning or redundant encoded columns
may be contributing; scaling alone is not proof of a modeling improvement.

Ridge uses a penalty on coefficient size. Its effect should be judged from the
current CV results, not the scores saved before removing `Id` and recoding
`MSSubClass`.


In [37]:
ridge_alphas = [
    0.01,
    0.1,
    1.0,
    10.0,
    30.0,
    100.0,
    300.0,
    1000.0
]

In [38]:
ridge_results = []

for alpha in ridge_alphas:
    ridge_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor_scaled),
            ("model", Ridge(alpha=alpha))
        ]
    )

    scores = -cross_val_score(
        ridge_model,
        X_train,
        y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    ridge_results.append({
        "alpha": alpha,
        "mean_cv_rmse": scores.mean(),
        "std_cv_rmse": scores.std()
    })

In [39]:
ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df.sort_values("mean_cv_rmse")

,alpha,mean_cv_rmse,std_cv_rmse
4,30.00,0.139579,0.030654
3,10.00,0.140285,0.029221
5,100.00,0.141731,0.031696
6,300.00,0.146436,0.030771
2,1.00,0.147924,0.027452
7,1000.00,0.158845,0.025920
1,0.10,0.158903,0.025699
0,0.01,0.163565,0.024570


### Coarse Ridge Alpha Sweep

The table above compares several Ridge regularization strengths (`alpha`).
The best candidate in this sweep has the **lowest mean CV RMSE** among the
listed values. Small differences may reflect variability across the five folds,
so they do not establish a uniquely optimal regularization strength.


### Interpreting the Coarse Search

Very small `alpha` values resemble a weakly regularized linear model; very
large values shrink coefficients more strongly and may underfit. The score
curve, rather than a fixed assumption about the best `alpha`, should guide
the next narrower search.


### Fine Ridge Search

A second search tests `alpha` values of 15, 20, 30, 40, 50, and 70.
The best-performing value from this **predefined grid** will be selected
programmatically below. Reusing the same CV folds for hyperparameter selection
can make the selected CV score optimistic; the reserved holdout must not be
used to choose hyperparameters.


In [40]:
ridge_alphas_fine = [
    15,
    20,
    30,
    40,
    50,
    70
]

In [41]:
ridge_fine_results = []

for alpha in ridge_alphas_fine:
    ridge_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor_scaled),
            ("model", Ridge(alpha=alpha))
        ]
    )

    scores = -cross_val_score(
        ridge_model,
        X_train,
        y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    ridge_fine_results.append({
        "alpha": alpha,
        "mean_cv_rmse": scores.mean(),
        "std_cv_rmse": scores.std()
    })

In [42]:
ridge_fine_results_df = pd.DataFrame(
    ridge_fine_results
)

ridge_fine_results_df.sort_values("mean_cv_rmse")

,alpha,mean_cv_rmse,std_cv_rmse
1,20,0.139543,0.030124
2,30,0.139579,0.030654
0,15,0.139744,0.029718
3,40,0.139825,0.031011
4,50,0.140139,0.031255
5,70,0.140785,0.031531


### Fine Ridge Search Result

The fine-search table above reports the mean and standard deviation of RMSE
for every tested `alpha`. The candidate chosen below is the one with the
lowest **mean** CV RMSE in that table; ties favor the first table entry.

Treat close CV scores as approximately equivalent rather than claiming that
tiny improvements prove one `alpha` is universally best. This is still
model selection on the training set, **not final evaluation**.


In [43]:
best_fine_idx = ridge_fine_results_df["mean_cv_rmse"].idxmin()
selected_ridge_alpha = float(ridge_fine_results_df.loc[best_fine_idx, "alpha"])

selected_ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", Ridge(alpha=selected_ridge_alpha))
    ]
)

print(f"Selected Ridge alpha (fine grid): {selected_ridge_alpha:g}")


Selected Ridge alpha (fine grid): 20


In [44]:
selected_fine_result = ridge_fine_results_df.loc[best_fine_idx]

experiment_results = pd.DataFrame({
    "experiment": [
        "Baseline Linear",
        "Numerical Missing Indicators",
        "Neighborhood LotFrontage",
        "Scaled Linear",
        "Ridge alpha=1",
        f"Ridge alpha={selected_ridge_alpha:g} (selected)"
    ],
    "mean_cv_rmse": [
        cv_rmse.mean(),
        cv_rmse_with_indicator.mean(),
        cv_rmse_neighborhood_frontage.mean(),
        cv_rmse_linear_scaled.mean(),
        cv_rmse_ridge.mean(),
        selected_fine_result["mean_cv_rmse"]
    ],
    "std_cv_rmse": [
        cv_rmse.std(),
        cv_rmse_with_indicator.std(),
        cv_rmse_neighborhood_frontage.std(),
        cv_rmse_linear_scaled.std(),
        cv_rmse_ridge.std(),
        selected_fine_result["std_cv_rmse"]
    ]
})

experiment_results.sort_values("mean_cv_rmse").reset_index(drop=True)


,experiment,mean_cv_rmse,std_cv_rmse
0,Ridge alpha=20 (selected),0.139543,0.030124
1,Ridge alpha=1,0.147924,0.027452
2,Baseline Linear,0.165385,0.023934
3,Neighborhood LotFrontage,0.165386,0.024171
4,Numerical Missing Indicators,0.165507,0.024255
5,Scaled Linear,0.168524,0.026301


## Preprocessing Summary

This notebook evaluates the following *training-fold-fitted* strategies:

- Median imputation for numerical features and a dedicated `"Missing"` category
  for categorical features (baseline).
- Numerical missingness indicators.
- Neighborhood-based `LotFrontage` imputation with a training-fold global fallback.
- Standardization of numerical features and Ridge regularization.
- A coarse and fine grid search for Ridge's `alpha` parameter.

`Id` is excluded from the predictors and `MSSubClass` is encoded as categorical.
**Compare the freshly computed table above** to draw conclusions about which
changes actually improved CV RMSE. Previous numeric results are intentionally
not quoted, because changing feature handling changes the experiments.

The selected Ridge pipeline is **an unfitted candidate** for later comparison.
Its `alpha` comes from the current training CV fine search. No preprocessing
rule, hyperparameter, or model was selected using the 20% holdout set.

The candidate combines neighborhood-based frontage imputation, median imputation
of other numerical features, numerical standardization, categorical missing-value
handling, one-hot encoding, and Ridge regression. Its final performance must be
assessed independently after model selection is complete.


### Numerical Solver Stability

A tolerance-sensitivity check found that the cross-validation results of sparse ordinary least-squares models changed when the iterative solver tolerance was tightened. For a consistent comparison, all `LinearRegression` experiments now use `tol=1e-10`. This controls solver convergence, not model regularization. The results should be recomputed after this change; no conclusion about the intrinsic benefit or harm of feature scaling is drawn from the earlier scores.

Note that selecting the Ridge `alpha` using the same cross-validation scores can make its reported minimum slightly optimistic. Final model evaluation remains reserved for the untouched holdout set.